In [1]:
import os
import sys
import pprint
from pathlib import Path
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat, RoundRobinGroupChat
from autogen_agentchat.tools import AgentTool
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import StdioServerParams, mcp_server_tools

# Project import paths
# Get the current directory (which is TFM_BA_GPT/test)
notebook_dir = os.getcwd()

# Define base project directory (one level up from test/)
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))

# Server root: one level up from project_root
code_root = os.path.abspath(os.path.join(project_root, '..'))
server_root = os.path.join(project_root, 'mcp_servers')



# Define paths to each server and Python executable
PyMCP_FS_path = os.path.join(server_root, 'PyMCP-FS')
Mermaid_MCP_path = os.path.join(server_root, 'Mermaid-MCP')
PDF_format_path = os.path.join(server_root, 'Final_deliverable_MCP_server')
tmp_dir = os.path.join(server_root, 'tmp')

# Python executables
PyMCP_FS_python = os.path.join(PyMCP_FS_path, '.venv', 'bin', 'python')
Mermaid_MCP_python = os.path.join(Mermaid_MCP_path, 'venv', 'bin', 'python')
PDF_format_python = os.path.join(PDF_format_path, 'venv', 'bin', 'python')




# Compute absolute paths for lib and src (one level up from test/)
lib_path = os.path.abspath(os.path.join(notebook_dir, '..', 'lib'))
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))

sys.path.append(lib_path)
sys.path.append(src_path)

from model_loader import load_model_client

In [2]:
model_client = load_model_client("azure")

In [3]:
# Helpful sanity prints:
print("raw:", getattr(model_client, "_raw_config", None))
print("create_args:", getattr(model_client, "_create_args", None))

raw: {'model': 'gpt-4o', 'api_key': 'REPLACE_WITH_YOUR_API_KEY', 'azure_endpoint': 'https://224d2-m9kjdmy0-eastus2.openai.azure.com/', 'azure_deployment': 'gpt-4o', 'api_version': '2024-12-01-preview'}
create_args: {'model': 'gpt-4o'}


In [4]:
# Server params
PyMCP_FS = StdioServerParams(
    command="/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/bin/python",
    args=[
        "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/mcp_servers/PyMCP-FS/main.py",
        "-d", project_root,      # ✅
        "-d", "/tmp",            # ✅ was missing the -d
        "--log-level", "DEBUG",
        "--log-file", "mcp_debug.log",
        "--project-root", project_root,
    ],
)

In [5]:


Mermaid_MCP = StdioServerParams(
    command="uv",
    args=[
        "run",
        os.path.join(Mermaid_MCP_path, 'Mermaid-MCP.py'),
        '-d', project_root,
        '-d', tmp_dir,
        '--project-root', project_root
    ]
)

PDF_format_mcp_server = StdioServerParams(
    command="uv",
    args=[
        "run",
        os.path.join(PDF_format_path, 'PDF_format_mcp_server.py'),
        '-d', project_root,
        '--project-root', project_root,
    ],
    read_timeout_seconds=60.0,
)


In [6]:
PyMCP_FS_mcp_tools = await mcp_server_tools(PyMCP_FS)

In [7]:
mermaid_mcp_tools = await mcp_server_tools(Mermaid_MCP)

In [8]:
pdf_format_mcp_tools = await mcp_server_tools(PDF_format_mcp_server)

In [9]:

# ---- terminate_agent system message (MOVE + clean + render_mermaid_all + robust PDF) ----
terminate_system_message = r"""
You are the TERMINATE_AGENT. Perform the final export pipeline and then end with **TERMINATE**.

**INPUTS**
  conversation log: project_root/conversation/interview.jsonl
  spec:             project_root/conversation/functional_specification.md
  mermaid:          project_root/conversation/process_diagram.mmd

**GOAL**
  Create: project_root/experience/<SESSION_ID>/
      interview.jsonl
      functional_specification.md
      process_diagram.mmd
      process_diagram.png  
      functional_specification.formatted.md
      deliverable.pdf

**STEPS STRICT ORDER**
  1 Read project_root/conversation/interview.jsonl and extract the first available "session_id".
    If multiple lines: pick the first non-empty "session_id".
    If not found: set SESSION_ID = "sess-unknown".

  2 Create folder project_root/experience/<SESSION_ID>/

  3 MOVE the three source files into that folder using 'move_file':
      project_root/conversation/interview.jsonl             -> project_root/experience/<SESSION_ID>/interview.jsonl
      project_root/conversation/functional_specification.md -> project_root/experience/<SESSION_ID>/functional_specification.md
      project_root/conversation/process_diagram.mmd         -> project_root/experience/<SESSION_ID>/process_diagram.mmd

  4 Clean the Mermaid file **in place** and report issues (severity/code/message/recommendation):
      clean_mermaid(
        input_path="project_root/experience/<SESSION_ID>/process_diagram.mmd",
        output_path="project_root/experience/<SESSION_ID>/process_diagram.mmd",
        overwrite=True
      )

  5 Render Mermaid **PNG**:
      render_mermaid_png(
        input_path="project_root/experience/<SESSION_ID>/process_diagram.mmd",
        output_path="project_root/experience/<SESSION_ID>/process_diagram.png",
        theme="forest"
      )

  6 Run **format_and_export** ONLY:
      format_and_export(
        session_dir="project_root/experience/<SESSION_ID>",
        md="functional_specification.md",
        diagram="process_diagram.png",
        md_out="functional_specification.formatted.md",
        pdf_out="deliverable.pdf",
        engine="xelatex",
        toc=True, toc_depth=2, margin="1in",
        mainfont="DejaVu Serif", monofont="DejaVu Sans Mono"
      )

  7 Confirm project_root/experience/<SESSION_ID>/deliverable.pdf exists.

  8 Save experience metadata:
      save_experience_chroma(
        session_dir="project_root/experience/<SESSION_ID>",
        session_id="<SESSION_ID>"
      )

  9 Output exactly: **TERMINATE**

**RULES**
  Be explicit about each steps success or failure.
  If a tool call fails, retry up to 1 times with the same arguments.
"""


In [10]:
termination = TextMentionTermination("TERMINATE")

terminate_agent = AssistantAgent(
    name="terminate_agent",
    model_client=model_client,
    system_message=terminate_system_message,
    tools=(PyMCP_FS_mcp_tools + mermaid_mcp_tools + pdf_format_mcp_tools),
    reflect_on_tool_use=False,
)
team = RoundRobinGroupChat(
    [terminate_agent],
    termination_condition=termination,
    )
await Console(team.run_stream(task="Execute the termination pipeline"))

---------- TextMessage (user) ----------
Execute the termination pipeline
---------- ToolCallRequestEvent (terminate_agent) ----------
[FunctionCall(id='call_wMjTJPj6jrQvpHKOXN7AYZLV', arguments='{"path":"project_root/conversation/interview.jsonl"}', name='read_file')]


/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/autogen_agentchat/agents/_assistant_agent.py:1107: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


---------- ToolCallExecutionEvent (terminate_agent) ----------
[FunctionExecutionResult(content='[{"type": "text", "text": "{\\"_meta\\": {\\"session_id\\": \\"sess-20231007XXXXXX\\", \\"ts\\": \\"2025-09-10T18:48:49.624869Z\\", \\"n_records\\": 2}}\\n{\\"speaker\\": \\"interviewer\\", \\"message\\": \\"QUESTION: Can you provide an overview of the product, its purpose, and the main problems it aims to solve?\\", \\"subject\\": \\"Product Overview\\", \\"q_index\\": 1, \\"timestamp\\": \\"2025-09-10T18:48:49.624838Z\\"}\\n{\\"speaker\\": \\"user\\", \\"message\\": \\"The scope of the project is to design and develop an online ecommerce website for apparels. Customers will be able to use the frontend website for searching the apparels and place the order by making online payment of the order in advance. Web based admin panel/ backend will help admin user i.e. business owner to manage the products, categories, prices and orders placed by the customers.\\", \\"subject\\": \\"Product Overvi

/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/autogen_agentchat/agents/_assistant_agent.py:1107: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


---------- ToolCallExecutionEvent (terminate_agent) ----------
[FunctionExecutionResult(content='[{"type": "text", "text": "Successfully created directory project_root/experience/sess-20231007XXXXXX", "annotations": null}]', name='create_directory', call_id='call_AAQsoU4VPHjn9pDSWlQhDvWX', is_error=False)]
---------- ToolCallSummaryMessage (terminate_agent) ----------
[{"type": "text", "text": "Successfully created directory project_root/experience/sess-20231007XXXXXX", "annotations": null}]
---------- ToolCallRequestEvent (terminate_agent) ----------
[FunctionCall(id='call_Pxkw1jgWhUs0FTnmx6ZWxIxz', arguments='{"source": "project_root/conversation/interview.jsonl", "destination": "project_root/experience/sess-20231007XXXXXX/interview.jsonl"}', name='move_file'), FunctionCall(id='call_aAqJ1EfaI4yo3dCXPZIus5J4', arguments='{"source": "project_root/conversation/functional_specification.md", "destination": "project_root/experience/sess-20231007XXXXXX/functional_specification.md"}', name='

/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/autogen_agentchat/agents/_assistant_agent.py:1107: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


... (9 outputs left out)

/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/autogen_agentchat/agents/_assistant_agent.py:1107: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


---------- ToolCallExecutionEvent (terminate_agent) ----------
[FunctionExecutionResult(content='[{"type": "text", "text": "{\\"ok\\":true,\\"session_id\\":\\"sess-20231007XXXXXX\\",\\"files_indexed\\":0,\\"written_items\\":2,\\"collection\\":\\"tfm_ba_gpt_experiences\\",\\"persistence_path\\":\\"/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.experiences\\"}", "annotations": null}]', name='save_experience_chroma', call_id='call_jq79QyV3l0wP750zHGsdGAUD', is_error=False)]
---------- ToolCallSummaryMessage (terminate_agent) ----------
[{"type": "text", "text": "{\"ok\":true,\"session_id\":\"sess-20231007XXXXXX\",\"files_indexed\":0,\"written_items\":2,\"collection\":\"tfm_ba_gpt_experiences\",\"persistence_path\":\"/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.experiences\"}", "annotations": null}]
---------- TextMessage (terminate_agent) ----------
**TERMINATE**


/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/autogen_agentchat/agents/_assistant_agent.py:1107: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


TaskResult(messages=[TextMessage(id='da3a0a39-5dd9-4021-9636-c856ccfa7ec9', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 9, 10, 22, 6, 26, 843657, tzinfo=datetime.timezone.utc), content='Execute the termination pipeline', type='TextMessage'), ToolCallRequestEvent(id='a1e5ec35-bc21-4b1e-b464-2a6ccc9cbb4d', source='terminate_agent', models_usage=RequestUsage(prompt_tokens=3288, completion_tokens=22), metadata={}, created_at=datetime.datetime(2025, 9, 10, 22, 6, 28, 101811, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_wMjTJPj6jrQvpHKOXN7AYZLV', arguments='{"path":"project_root/conversation/interview.jsonl"}', name='read_file')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='5de88f32-c111-47e9-a562-02f106dbee0d', source='terminate_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 9, 10, 22, 6, 28, 433134, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='[{"type": "text", "text"